# 454. 4Sum II

[Problem](https://leetcode.com/problems/4sum-ii/) · difficulty: medium

Eight approaches that converge on the same O(n²) meet-in-the-middle idea.
This notebook traces the key invariant on a small input, exposes the
pruning step in `SolutionSplitPruned`, and benchmarks the full progression.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0454-4sum-ii'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
[s.__name__ for s in solutions]

## The invariant: meet in the middle

The four arrays are **independent** — any element from `nums1` can pair with any
element from `nums2`, regardless of what `nums3` or `nums4` contain. That independence
lets us split the problem:

$$
a + b + c + d = 0 \iff a + b = -(c + d)
$$

Build a frequency map of every pairwise sum $s = a + b$ (counting
multiplicities). For every pair $(c, d)$, look up $-(c+d)$ in that map.
Each hit contributes its stored count to the total. Two O(n²) passes replaces
one O(n⁴) scan.


## Trace on a small input

The problem's own Example 1: `nums1=[1,2]`, `nums2=[-2,-1]`, `nums3=[-1,2]`,
`nums4=[0,2]`. Expected output: `2`.


In [ ]:
from collections import Counter

nums1, nums2, nums3, nums4 = [1, 2], [-2, -1], [-1, 2], [0, 2]

# --- Phase 1: build sum12 ---
sum12 = {}
for a in nums1:
    for b in nums2:
        s = a + b
        sum12[s] = sum12.get(s, 0) + 1
        print(f'  a={a:+d}  b={b:+d}  a+b={s:+d}  sum12={dict(sum12)}')

print()
print('sum12 (count of each a+b sum):', sum12)
print()

# --- Phase 2: look up -(c+d) in sum12 ---
total = 0
for c in nums3:
    for d in nums4:
        need = -c - d
        hit  = sum12.get(need, 0)
        total += hit
        print(f'  c={c:+d}  d={d:+d}  need=-(c+d)={need:+d}  hit={hit}  running_total={total}')

print()
print('Answer:', total)

The trace shows exactly which `(c, d)` pairs find a matching `a+b` sum and why
the stored count is what gets added — three elements from `nums1 × nums2` that
produce the same sum would contribute 3 to every matching `(c, d)` pair.


## Why `list.count()` inside a loop hurts

Classes `SolutionSplitPairCount` through `SolutionSplitSetCount` all call `list.count()` to
build per-value frequencies. `list.count()` is a linear scan — O(n) — so
calling it once per unique value turns the build step into O(n · u) where
`u` is the number of unique values. `Counter` does the same work in a single
O(n) pass.


In [ ]:
import time

big = list(range(-500, 500))   # 1000 elements, all distinct (worst case for .count())

def build_freq_count(lst):
    freq = {}
    for v in set(lst):
        freq[v] = lst.count(v)   # O(n) per unique value
    return freq

def build_freq_counter(lst):
    return dict(Counter(lst))    # O(n) total

RUNS = 500
for label, fn in [('list.count() loop', build_freq_count), ('Counter', build_freq_counter)]:
    t0 = time.perf_counter()
    for _ in range(RUNS):
        fn(big)
    elapsed_us = (time.perf_counter() - t0) / RUNS * 1e6
    print(f'{label:<22}  {elapsed_us:6.1f} µs per call')

## The pruning step in `SolutionSplitPruned`

`SolutionSplitPruned` skips building `sums34` entries whose negated sum is
absent from `sums12` — if `-(c+d)` is not a key in `sums12`, that `(c, d)`
pair can never contribute to the total, so it is skipped with `continue`:

```python
for c in vals3:
    for d in vals4:
        need = -c - d
        if need not in sums12:
            continue   # ← skip only this (c, d) pair; d-loop continues
        sums34[need] = sums34.get(need, 0) + freq3[c] * freq4[d]
```

The entries skipped are exactly those the final join would have ignored anyway
(no matching `a+b` sum exists), so the result is correct. The code below
confirms every solution agrees on both large test cases.


In [ ]:
# Run every solution on both large test cases from test_4sum_ii.py
# and show which ones produce the wrong answer.

import importlib
test_mod_path = str(PROBLEM / 'test_4sum_ii.py')
spec = importlib.util.spec_from_file_location('_test_454', test_mod_path)
test_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_mod)

# Cases 2 and 3 are the large inputs (indices 2 and 3 after the two tiny ones)
large_cases = test_mod.CASES[2:]

print(f'{'Solution':<22}  ', end='')
for args, expected in large_cases:
    print(f'  n={len(args[0]):>3} (exp {expected:>7})', end='')
print()

for sol_cls in solutions:
    print(f'{sol_cls.__name__:<22}', end='')
    for args, expected in large_cases:
        got = sol_cls().fourSumCount(*args)
        mark = '✓' if got == expected else f'✗ got {got}'
        print(f'  {mark:>22}', end='')
    print()

## Benchmark: full progression

All eight solutions on three input shapes at `n = 200` (the constraint's limit).


In [ ]:
import random

random.seed(454)

def make_arrays(n, lo=-200, hi=200):
    return tuple([random.randint(lo, hi) for _ in range(n)] for _ in range(4))

SHAPES = {
    'n=50,  dense (±10)':  make_arrays(50,  -10, 10),
    'n=200, dense (±20)':  make_arrays(200, -20, 20),
    'n=200, sparse (±200)': make_arrays(200, -200, 200),
}

WARMUP = 2
RUNS   = 10

def timed(sol_cls, arrays, runs=RUNS, warmup=WARMUP):
    for _ in range(warmup):
        sol_cls().fourSumCount(*arrays)
    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        sol_cls().fourSumCount(*arrays)
        samples.append((time.perf_counter() - t0) * 1e3)
    return min(samples)

header = f"{'Solution':<22}" + ''.join(f'{label:>26}' for label in SHAPES)
print(header)
print('-' * len(header))
for sol_cls in solutions:
    row = ''.join(f'{timed(sol_cls, arrays):23.2f} ms' for arrays in SHAPES.values())
    print(f'{sol_cls.__name__:<22}{row}')

## Summary

| # | Approach | Time | What changed |
|---|---|---|---|
| 1 | `SolutionTripleLoop` | O(u³) | Baseline triple loop |
| 2 | `SolutionSplitPairCount` | O(u²·n) | Split; `.count()` called once per unique *pair* |
| 3 | `SolutionSplitAllCount` | O(n²) | `.count()` called once per element index |
| 4 | `SolutionSplitGuardedCount` | O(n·u + u²) | `.count()` guarded; called once per unique value |
| 5 | `SolutionSplitSetCount` | O(n·u + u²) | Explicit `set()` upfront; same bound, cleaner |
| 6 | `SolutionSplitPruned` | O(n·u + u²) | Skips `sums34` entries absent from `sums12` |
| 7 | `SolutionSplitDirect` | O(n·u + u²) | No `sums34` dict; accumulates directly |
| 8 | `SolutionSplitCounter` | O(n²) | `Counter` replaces manual loops; cleanest form |

Takeaways:

- **`Counter`** costs the same as a manual single-pass frequency map but
  removes four lines of boilerplate per array.
- The big jump (**`SolutionSplitPairCount` → `SolutionSplitAllCount`**) is
  13.0× → 2.0× — not an algorithmic change, but moving `.count()` from
  once-per-*pair* to once-per-*value*.
- Meet-in-the-middle works whenever the input splits into independent groups;
  the two-phase approach here drops O(n⁴) to O(n²) for free.
